In [ ]:
code = 'HEDGED_STRADDLE_SEPARATE_LEG'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def HEDGED_STRADDLE_SEPARATE_LEG(bt, start_time, end_time, sell_om, hedge_pct, sl, target):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        sell_ce_scrip, sell_pe_scrip, _, _, future_price, entry_dt = bt.get_strike(start_dt, end_dt, om=sell_om)
        if sell_ce_scrip is None: return None

        ### hedge sits hedge_pct away from each short leg's own strike, rounded onto the strike grid
        sell_ce_strike, sell_pe_strike = get_strike(sell_ce_scrip), get_strike(sell_pe_scrip)
        ce_offset = round((sell_ce_strike * (hedge_pct/100)) / bt.gap) * bt.gap
        pe_offset = round((sell_pe_strike * (hedge_pct/100)) / bt.gap) * bt.gap
        if (ce_offset <= 0) or (pe_offset <= 0): return None

        hedge_ce_scrip = f"{sell_ce_strike + ce_offset}CE"
        hedge_pe_scrip = f"{sell_pe_strike - pe_offset}PE"

        ### each side stands alone - matched only against its own hedge, with its own credit, sl and target
        def run_side(sell_scrip, hedge_scrip):

            sell_data = bt.get_single_leg_data(entry_dt, end_dt, sell_scrip)
            hedge_data = bt.get_single_leg_data(entry_dt, end_dt, hedge_scrip)

            common_dt = np.intersect1d(sell_data['date_time'].values, hedge_data['date_time'].values)
            if len(common_dt) == 0:
                return None

            sell_data = sell_data[np.isin(sell_data['date_time'].values, common_dt)]
            hedge_data = hedge_data[np.isin(hedge_data['date_time'].values, common_dt)]

            side_entry_time = sell_data['date_time'].iloc[0]
            sell_entry = sell_data['close'].iloc[0]
            hedge_entry = hedge_data['close'].iloc[0]

            side_credit = sell_entry - hedge_entry
            if side_credit <= 0:
                return None

            value_list = (sell_data['close'].values - hedge_data['close'].values).tolist()

            sl_price = side_credit * (1 + (sl/100)) if sl else None
            target_price = side_credit * (1 - (target/100)) if target else None

            exit_reason, exit_index = 'EOD', len(value_list) - 1

            for i, ele in enumerate(value_list if (sl_price is not None) or (target_price is not None) else []):
                if (sl_price is not None) and (ele >= sl_price):
                    exit_reason, exit_index = 'SL', i
                    break
                if (target_price is not None) and (ele <= target_price):
                    exit_reason, exit_index = 'TARGET', i
                    break

            side_exit_time = sell_data['date_time'].iloc[exit_index]
            side_exit_value = value_list[exit_index]

            sell_exit = sell_data['close'].iloc[exit_index]
            hedge_exit = hedge_data['close'].iloc[exit_index]

            ### slipage is charged per leg on its own entry premium
            sell_slipage = round(bt.Cal_slipage(sell_entry), 2)
            hedge_slipage = round(bt.Cal_slipage(hedge_entry), 2)

            sell_pnl = round((sell_entry - sell_exit) - sell_slipage, 2)
            hedge_pnl = round((hedge_exit - hedge_entry) - hedge_slipage, 2)

            side_slipage = round(sell_slipage + hedge_slipage, 2)
            side_pnl = round(sell_pnl + hedge_pnl, 2)

            return [sell_scrip, sell_entry, sell_exit, sell_slipage, sell_pnl, hedge_scrip, hedge_entry, hedge_exit, hedge_slipage, hedge_pnl, side_entry_time.time(), side_credit, side_exit_value, exit_reason, side_exit_time.time(), side_slipage, side_pnl]

        ce_side = run_side(sell_ce_scrip, hedge_ce_scrip)
        if ce_side is None: return None

        pe_side = run_side(sell_pe_scrip, hedge_pe_scrip)
        if pe_side is None: return None

        credit = round(ce_side[11] + pe_side[11], 2)
        slipage = round(ce_side[15] + pe_side[15], 2)
        total_pnl = round(ce_side[16] + pe_side[16], 2)

        return [code, bt.index, start_time, end_time, sell_om, hedge_pct, sl, target, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_dt.time(), future_price, sell_ce_strike, sell_pe_strike, ce_offset, pe_offset] + ce_side + pe_side + [credit, slipage, total_pnl]

    except Exception as e:
        print(e, [bt.index, bt.current_date.date(), bt.current_date.day_name(), start_time, end_time, sell_om, hedge_pct, sl, target])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_SellOM/P_HedgePct/P_SL/P_Target/Date/Day/DTE/EntryTime/Future/Sell.CE.Strike/Sell.PE.Strike/CE.Hedge.Offset/PE.Hedge.Offset/Sell.CE/Sell.CE.Entry/Sell.CE.Exit/Sell.CE.Slipage/Sell.CE.PNL/Hedge.CE/Hedge.CE.Entry/Hedge.CE.Exit/Hedge.CE.Slipage/Hedge.CE.PNL/CE.Entry.Time/CE.Credit/CE.Exit.Value/CE.Exit.Reason/CE.Exit.Time/CE.Slipage/CE.Net.PNL/Sell.PE/Sell.PE.Entry/Sell.PE.Exit/Sell.PE.Slipage/Sell.PE.PNL/Hedge.PE/Hedge.PE.Entry/Hedge.PE.Exit/Hedge.PE.Slipage/Hedge.PE.PNL/PE.Entry.Time/PE.Credit/PE.Exit.Value/PE.Exit.Reason/PE.Exit.Time/PE.Slipage/PE.Net.PNL/Credit/Slipage/Total.PNL'.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [HEDGED_STRADDLE_SEPARATE_LEG(bt, row.entry_time, row.exit_time, row.sell_om, row.hedge_pct, row.sl, row.target) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)

                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()

                    t2 = datetime.datetime.now()
                    print(t2-t1)

        except Exception as e:
            input(str(e))